# 🚀 ALTAIR: NIFTY 50 & W.D. Gann Cycle Harmonization Lab

Welcome to the official **ALTAIR Quantitative Research Lab**. This notebook combines **NIFTY 50 Market Analytics**, **W.D. Gann Time & Price Cycles**, and the **Swing Trade Engine (STE)**.

### 🎯 Key Lab Capabilities:
1. **NIFTY 50 Ingestion**: Pull daily OHLCV bars for `^NSEI`.
2. **W.D. Gann Time Cycles**: Project turning dates from structural highs and lows using Gann's harmonic cycle degrees ($30, 45, 60, 90, 120, 135, 180, 225, 270, 315, 360$ days).
3. **Cycle Confluence Scanner**: Identify **Resonance Clusters** where multiple cycle projections converge on the exact same turning window.
4. **Gann Square of 9 Price Levels**: Calculate mathematical geometric support & resistance angles ($45^\circ, 90^\circ, 180^\circ, 270^\circ, 360^\circ$).
5. **Turn Accuracy Backtest**: Measure historical accuracy—how reliably NIFTY 50 reverses near Gann cycle dates.
6. **Dual-Gate STE Stock Backtester**: Dynamic ATR-based swing trading engine for individual stock picks.
7. **Gemini AI Quant Copilot**: Chat directly with Gemini to audit cycles and fine-tune your parameters.

## 1. Environment Setup & Engine Imports
Link to ALTAIR engine modules and import scientific libraries.

In [1]:
import sys
import os
import math
from datetime import datetime, timedelta
import numpy as np
import pandas as pd
import yfinance as yf
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import matplotlib.pyplot as plt

# Add project root to path
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
    sys.path.insert(0, os.path.join(PROJECT_ROOT, 'backend'))

print(f"[+] Project root linked: {PROJECT_ROOT}")

# Check backend modules
try:
    from backend.src.engine.swing.SwingScanner import classify_swing_setup
    from backend.src.engine.swing.TechnicalScore import _rsi, _up_down_volume_ratio
    print("[+] Successfully imported ALTAIR Swing Engine components!")
except ImportError as e:
    print(f"[*] Notice on backend imports: {e}")

[+] Project root linked: D:\ANTI-GRAVITY\ALTAIR BASE
[+] Successfully imported ALTAIR Swing Engine components!


## 2. Gemini AI Quant Copilot
Connect to Gemini to analyze Gann turning dates and backtest outputs.

In [2]:
from google import genai

gemini_api_key = os.environ.get('GEMINI_API_KEY')
if not gemini_api_key:
    env_path = os.path.join(PROJECT_ROOT, 'backend', '.env')
    if os.path.exists(env_path):
        with open(env_path, 'r') as f:
            for line in f:
                if line.startswith('GEMINI_API_KEY=') or line.startswith('GOOGLE_API_KEY='):
                    gemini_api_key = line.split('=', 1)[1].strip().strip('"').strip("'")
                    break

client = None
if gemini_api_key:
    client = genai.Client(api_key=gemini_api_key)
    print("[+] Gemini Client connected successfully!")
else:
    print("[*] To activate Gemini Copilot: set os.environ['GEMINI_API_KEY'] = 'your_key'")

def ask_gemini(prompt: str, context_data: dict = None) -> str:
    """Interact with Gemini directly inside the research lab."""
    if not client:
        return "Configure GEMINI_API_KEY to activate AI responses."
    
    full_prompt = f"You are the Principal Quant Strategist for the ALTAIR Engine.\n\nQuery: {prompt}"
    if context_data:
        import json
        full_prompt += f"\n\nContext Data:\n{json.dumps(context_data, indent=2, default=str)}"
        
    response = client.models.generate_content(
        model='gemini-2.0-flash',
        contents=full_prompt
    )
    return response.text

[*] To activate Gemini Copilot: set os.environ['GEMINI_API_KEY'] = 'your_key'


## 3. Pull NIFTY 50 Daily OHLCV Data & Key Indicators
Download historical data for NIFTY 50 (`^NSEI`) and calculate Moving Averages, RSI(14), and ATR(14).

In [3]:
# Download 2 years of daily NIFTY 50 bars
print("[*] Downloading NIFTY 50 historical data...")
nifty_raw = yf.download("^NSEI", period="2y", interval="1d", progress=False)
if isinstance(nifty_raw.columns, pd.MultiIndex):
    nifty_raw.columns = [col[0] for col in nifty_raw.columns]

nifty = nifty_raw.copy()

# Moving Averages
nifty['MA_20'] = nifty['Close'].rolling(window=20).mean()
nifty['MA_50'] = nifty['Close'].rolling(window=50).mean()
nifty['MA_200'] = nifty['Close'].rolling(window=200).mean()

# Wilder's RSI (14)
delta = nifty['Close'].diff()
gain = delta.clip(lower=0)
loss = -delta.clip(upper=0)
avg_gain = gain.ewm(alpha=1/14, min_periods=14, adjust=False).mean()
avg_loss = loss.ewm(alpha=1/14, min_periods=14, adjust=False).mean()
rs = avg_gain / avg_loss.replace(0, np.nan)
nifty['RSI'] = 100 - (100 / (1 + rs))

# Average True Range (14)
tr = pd.concat([
    nifty['High'] - nifty['Low'],
    (nifty['High'] - nifty['Close'].shift()).abs(),
    (nifty['Low'] - nifty['Close'].shift()).abs()
], axis=1).max(axis=1)
nifty['ATR'] = tr.rolling(window=14).mean()

print(f"[+] Successfully loaded {len(nifty)} daily bars for NIFTY 50!")
print(f"    Latest Close:  {nifty['Close'].iloc[-1]:.2f}")
print(f"    Current 50 DMA: {nifty['MA_50'].iloc[-1]:.2f}")
print(f"    Current 200 DMA:{nifty['MA_200'].iloc[-1]:.2f}")
print(f"    Current RSI:   {nifty['RSI'].iloc[-1]:.2f}")
print(f"    Current ATR:   {nifty['ATR'].iloc[-1]:.2f} pts")

nifty.tail(5)

[*] Downloading NIFTY 50 historical data...


[+] Successfully loaded 494 daily bars for NIFTY 50!
    Latest Close:  24080.40
    Current 50 DMA: 24206.29
    Current 200 DMA:24654.87
    Current RSI:   43.12
    Current ATR:   155.59 pts


,Close,High,Low,Open,Volume,MA_20,MA_50,MA_200,RSI,ATR
Date,,,,,,,,,,
2026-08-24,24219.050781,24313.000000,24144.300781,24285.050781,236300,24381.692578,24193.951992,24684.261797,47.850435,137.307059
2026-08-25,24334.550781,24334.550781,24115.449219,24175.750000,239500,24399.152637,24203.565000,24677.324053,52.394288,140.125000
2026-08-26,24207.750000,24378.599609,24207.750000,24341.949219,244800,24397.030176,24207.936992,24669.546055,47.500998,147.121373
2026-08-27,24090.849609,24297.449219,24090.849609,24277.599609,323400,24385.715137,24208.040000,24662.012051,43.470231,153.789202
2026-08-31,24080.400391,24128.699219,23993.599609,24117.550781,0,24370.555176,24206.288008,24654.865557,43.118026,155.592773


## 4. W.D. Gann Structural Pivot Detection & Time Cycles
Identify major market tops and bottoms in NIFTY 50 and project forward Gann Time Harmonic Cycles:
- **Gann Vibrational Days**: 30, 45, 60, 90, 120, 135, 180, 225, 270, 315, 360 calendar days.

In [4]:
# Detect major structural swing points (pivots)
pivot_window = 15
nifty['Is_High'] = (nifty['High'] == nifty['High'].rolling(window=pivot_window*2+1, center=True).max())
nifty['Is_Low'] = (nifty['Low'] == nifty['Low'].rolling(window=pivot_window*2+1, center=True).min())

pivots = []
for dt, row in nifty[nifty['Is_High']].iterrows():
    pivots.append({'date': dt, 'type': 'HIGH', 'price': float(row['High'])})
for dt, row in nifty[nifty['Is_Low']].iterrows():
    pivots.append({'date': dt, 'type': 'LOW', 'price': float(row['Low'])})

pivots.sort(key=lambda x: x['date'])

# Gann Harmonic Time Vibrations (calendar days)
GANN_CYCLES = [30, 45, 60, 90, 120, 135, 180, 225, 270, 315, 360]

projections = []
for p in pivots:
    for c in GANN_CYCLES:
        proj_date = p['date'] + timedelta(days=c)
        projections.append({
            'anchor_date': p['date'].strftime('%Y-%m-%d'),
            'anchor_type': p['type'],
            'anchor_price': p['price'],
            'cycle_days': c,
            'projected_date': proj_date,
            'projected_date_str': proj_date.strftime('%Y-%m-%d')
        })

df_projections = pd.DataFrame(projections)
print(f"[+] Identified {len(pivots)} major pivots and calculated {len(df_projections)} Gann time cycle projections.")

# Display recent pivots
pd.DataFrame(pivots).tail(6)

[+] Identified 25 major pivots and calculated 275 Gann time cycle projections.


,date,type,price
19,2026-04-02,LOW,22182.550781
20,2026-04-21,HIGH,24601.699219
21,2026-06-08,LOW,23070.150391
22,2026-07-07,HIGH,24530.900391
23,2026-07-24,LOW,23606.300781
24,2026-08-03,HIGH,24774.300781


## 5. Gann Cycle Confluence Detection (Resonance Clusters)
When multiple Gann cycles from different market swing anchors converge on the **same week**, W.D. Gann called it a **Time Cluster**—a high-probability turning date for NIFTY 50.

In [5]:
# Cluster projections that fall within a +/- 3-day window
clusters = []
tolerance_days = 3

# Filter for upcoming or recent active dates
today = datetime.now()
active_proj = [p for p in projections if abs((p['projected_date'] - today).days) <= 120]

for i, p1 in enumerate(active_proj):
    matched = [p1]
    for j, p2 in enumerate(active_proj):
        if i != j and abs((p1['projected_date'] - p2['projected_date']).days) <= tolerance_days:
            matched.append(p2)
    if len(matched) >= 2:
        clusters.append({
            'date': p1['projected_date_str'],
            'confluence_count': len(matched),
            'cycles_converging': [f"{m['cycle_days']}d from {m['anchor_type']} ({m['anchor_date']})" for m in matched]
        })

df_clusters = pd.DataFrame(clusters).drop_duplicates(subset=['date']).sort_values(by='confluence_count', ascending=False)
print(f"[+] Found {len(df_clusters)} Gann Time Resonance Clusters around current horizon:")
df_clusters.head(8)

[+] Found 75 Gann Time Resonance Clusters around current horizon:


,date,confluence_count,cycles_converging
68,2026-09-05,7,"[60d from HIGH (2026-07-07), 315d from HIGH (2..."
32,2026-06-02,6,"[120d from LOW (2026-02-02), 225d from HIGH (2..."
62,2026-09-06,6,"[90d from LOW (2026-06-08), 315d from HIGH (20..."
16,2026-06-05,5,"[225d from HIGH (2025-10-23), 360d from LOW (2..."
41,2026-08-02,5,"[180d from HIGH (2026-02-03), 360d from LOW (2..."
33,2026-06-17,5,"[135d from LOW (2026-02-02), 315d from LOW (20..."
34,2026-08-01,5,"[180d from LOW (2026-02-02), 360d from LOW (20..."
18,2026-09-03,5,"[315d from HIGH (2025-10-23), 135d from HIGH (..."


## 6. W.D. Gann Square of 9 Price Level Calculator
Calculates mathematical geometric support and resistance angles ($45^\circ, 90^\circ, 135^\circ, 180^\circ, 225^\circ, 270^\circ, 360^\circ$) from NIFTY 50's key anchor pivots:
$$\text{Target} = \left(\sqrt{\text{Price}} \pm \frac{\theta}{180}\right)^2$$

In [6]:
def gann_square_of_9(price: float) -> pd.DataFrame:
    """Calculates Gann Square of 9 angles for price targets and supports."""
    sqrt_p = math.sqrt(price)
    degrees = [45, 90, 135, 180, 225, 270, 315, 360]
    
    rows = []
    for deg in degrees:
        shift = deg / 180.0
        resistance = (sqrt_p + shift) ** 2
        support = (sqrt_p - shift) ** 2
        rows.append({
            'Angle': f"{deg}°",
            'Resistance_Level': round(resistance, 2),
            'Support_Level': round(support, 2),
            'Range_Pts': round(resistance - support, 2)
        })
    return pd.DataFrame(rows)

latest_pivot = pivots[-1]
df_sq9 = gann_square_of_9(latest_pivot['price'])

print(f"[*] Gann Square of 9 Harmonics from Anchor: {latest_pivot['type']} at {latest_pivot['price']:.2f} ({latest_pivot['date'].strftime('%Y-%m-%d')}):")
df_sq9

[*] Gann Square of 9 Harmonics from Anchor: HIGH at 24774.30 (2026-08-03):


,Angle,Resistance_Level,Support_Level,Range_Pts
0,45°,24853.06,24695.66,157.40
1,90°,24931.95,24617.15,314.80
2,135°,25010.96,24538.77,472.20
3,180°,25090.10,24460.50,629.59
4,225°,25169.36,24382.37,786.99
5,270°,25248.75,24304.36,944.39
6,315°,25328.26,24226.47,1101.79
7,360°,25407.89,24148.71,1259.19


## 7. Gann Cycle Turn Accuracy Backtest (Historical Hit-Rate)
Tests every historical Gann cycle projection against actual NIFTY 50 price action: did NIFTY reverse within $\pm 2$ trading days?

In [7]:
# Measure how many projected Gann dates matched an actual local inflection
hits = 0
total_evaluated = 0
tolerance_bars = 2

nifty_dates = set(nifty.index.strftime('%Y-%m-%d'))
pivot_dates = [p['date'] for p in pivots]

turn_verifications = []

for p in projections:
    proj_dt = p['projected_date']
    # Only evaluate projections that have occurred in the past
    if proj_dt < nifty.index[-1] - timedelta(days=5):
        total_evaluated += 1
        # Check if any actual pivot occurred within +/- tolerance_bars
        matched_pivot = None
        for p_actual in pivot_dates:
            if abs((p_actual - proj_dt).days) <= tolerance_bars + 1:
                matched_pivot = p_actual
                break
        
        is_hit = matched_pivot is not None
        if is_hit:
            hits += 1
            turn_verifications.append({
                'gann_cycle': f"{p['cycle_days']}d",
                'anchor_date': p['anchor_date'],
                'projected_date': p['projected_date_str'],
                'actual_reversal_date': matched_pivot.strftime('%Y-%m-%d'),
                'status': 'ACCURATE HIT'
            })

hit_rate = (hits / total_evaluated * 100) if total_evaluated > 0 else 0.0

print("==================================================")
print("W.D. Gann Cycle Turn Accuracy on NIFTY 50")
print("==================================================")
print(f"Total Cycle Dates Evaluated: {total_evaluated}")
print(f"Confirmed Reversal Hits:     {hits}")
print(f"Gann Cycle Turn Hit Rate:    {hit_rate:.2f}%")
print("==================================================")

pd.DataFrame(turn_verifications).tail(10)

W.D. Gann Cycle Turn Accuracy on NIFTY 50
Total Cycle Dates Evaluated: 205
Confirmed Reversal Hits:     52
Gann Cycle Turn Hit Rate:    25.37%


,gann_cycle,anchor_date,projected_date,actual_reversal_date,status
42,180d,2026-01-05,2026-07-04,2026-07-07,ACCURATE HIT
43,60d,2026-02-02,2026-04-03,2026-04-02,ACCURATE HIT
44,180d,2026-02-02,2026-08-01,2026-08-03,ACCURATE HIT
45,60d,2026-02-03,2026-04-04,2026-04-02,ACCURATE HIT
46,180d,2026-02-03,2026-08-02,2026-08-03,ACCURATE HIT
47,120d,2026-04-02,2026-07-31,2026-08-03,ACCURATE HIT
48,45d,2026-04-21,2026-06-05,2026-06-08,ACCURATE HIT
49,30d,2026-06-08,2026-07-08,2026-07-07,ACCURATE HIT
50,45d,2026-06-08,2026-07-23,2026-07-24,ACCURATE HIT
51,30d,2026-07-07,2026-08-06,2026-08-03,ACCURATE HIT


## 8. Interactive Plotly Visualizer: NIFTY 50 Candlesticks & Gann Cycle Grid
Visual chart with vertical Gann Time Turn lines and horizontal Square of 9 support/resistance levels.

In [8]:
fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.04, row_heights=[0.75, 0.25])

# 1. Candlestick Chart
fig.add_trace(go.Candlestick(
    x=nifty.index,
    open=nifty['Open'],
    high=nifty['High'],
    low=nifty['Low'],
    close=nifty['Close'],
    name="NIFTY 50"
), row=1, col=1)

# Moving averages
fig.add_trace(go.Scatter(x=nifty.index, y=nifty['MA_50'], line=dict(color='orange', width=1.5), name="50 DMA"), row=1, col=1)
fig.add_trace(go.Scatter(x=nifty.index, y=nifty['MA_200'], line=dict(color='cyan', width=1.5), name="200 DMA"), row=1, col=1)

# Draw Gann Square of 9 horizontal levels from latest pivot
for _, row_sq in df_sq9.head(4).iterrows():
    fig.add_hline(y=row_sq['Resistance_Level'], line_dash="dash", line_color="rgba(255, 0, 0, 0.4)", annotation_text=f"R: {row_sq['Angle']}", row=1, col=1)
    fig.add_hline(y=row_sq['Support_Level'], line_dash="dash", line_color="rgba(0, 255, 0, 0.4)", annotation_text=f"S: {row_sq['Angle']}", row=1, col=1)

# 2. RSI Subplot
fig.add_trace(go.Scatter(x=nifty.index, y=nifty['RSI'], line=dict(color='purple', width=1.5), name="RSI (14)"), row=2, col=1)
fig.add_hline(y=70, line_dash="dot", line_color="red", row=2, col=1)
fig.add_hline(y=30, line_dash="dot", line_color="green", row=2, col=1)

fig.update_layout(
    title="NIFTY 50: W.D. Gann Cycle & Square of 9 Harmonics",
    xaxis_rangeslider_visible=False,
    height=750,
    template="plotly_dark"
)
fig.show()

## 9. Gemini AI Analysis on Gann Cycles & Market Inflections
Ask Gemini to audit the current Gann cycle clusters and provide an outlook for NIFTY 50.

In [9]:
if client and not df_clusters.empty:
    summary_context = {
        'nifty_current_price': float(nifty['Close'].iloc[-1]),
        'nifty_50_dma': float(nifty['MA_50'].iloc[-1]),
        'nifty_200_dma': float(nifty['MA_200'].iloc[-1]),
        'current_rsi': float(nifty['RSI'].iloc[-1]),
        'gann_turn_hit_rate_pct': hit_rate,
        'upcoming_confluence_clusters': df_clusters.head(5).to_dict(orient='records'),
        'square_of_9_key_levels': df_sq9.head(4).to_dict(orient='records')
    }
    
    analysis = ask_gemini(
        "Analyze the current NIFTY 50 technical setup in relation to our Gann Time Confluence Clusters and Square of 9 levels. Which dates and price ranges should we watch closely for our swing trade engine entries?",
        context_data=summary_context
    )
    print("==================================================")
    print("🤖 Gemini Quant Copilot on NIFTY & Gann Cycles:")
    print("==================================================")
    print(analysis)
else:
    print("Set GEMINI_API_KEY to receive the automated AI market cycle audit.")

Set GEMINI_API_KEY to receive the automated AI market cycle audit.


## 9. Niche Sector & Sub-Field 4-Pillar Swing Scanner
This module evaluates companies grouped by granular industry fields (e.g., **Auto: 2-Wheelers**, **Auto: 4W & CV**, **Private Banks**, **PSU Banks**, **IT Services**, **Energy & Power**).

### 🏛️ The 4-Pillar Evaluation Model:
1. **Financial Health (25%)**: Debt/Equity leverage, ROE capital efficiency, Operating Margins, Astra Strike solvency.
2. **Technical Trend Momentum (30%)**: Price vs 20/50/200 DMA trend alignment and moving average crosses.
3. **RSI Momentum (20%)**: Wilder's RSI(14) sweet-spot momentum rebound (45-65 Continuation or <35 Oversold).
4. **W.D. Gann Price & Time Proximity (25%)**: Alignment with nearest Square of 9 geometric floor and cycle turning day.

Outputs a ranked **Swing Opportunity Leaderboard** with exact **Entry**, **Target (2.5x ATR)**, **Stop-Loss (1.5x ATR)**, and **Conviction Score**.

In [10]:
# Define Thematic Niche Industry Universes
SECTOR_NICHE_MAP = {
    "Adani Group - Top 6 Flagships": ["ADANIPORTS.NS", "AWL.NS", "ATGL.NS", "ADANIGREEN.NS", "ADANIENT.NS", "ADANIPOWER.NS"],
    "Auto - 2-Wheelers": ["BAJAJ-AUTO.NS", "TVSMOTOR.NS", "HEROMOTOCO.NS", "EICHERMOT.NS"],
    "Auto - 4W, Passenger & CV": ["TATAMOTORS.NS", "M&M.NS", "MARUTI.NS", "ASHOKLEY.NS", "BHARATFORG.NS"],
    "Banking - Private Banks": ["HDFCBANK.NS", "ICICIBANK.NS", "KOTAKBANK.NS", "AXISBANK.NS", "INDUSINDBK.NS"],
    "Banking - Public Sector (PSUs)": ["SBIN.NS", "BANKBARODA.NS", "PNB.NS", "CANBK.NS", "UNIONBANK.NS"],
    "IT - Tier 1 Enterprise Tech": ["TCS.NS", "INFY.NS", "HCLTECH.NS", "WIPRO.NS", "TECHM.NS"],
    "Energy - Upstream & Power Grid": ["RELIANCE.NS", "ONGC.NS", "NTPC.NS", "POWERGRID.NS", "TATAPOWER.NS"],
    "Consumer - FMCG & Discretionary": ["ITC.NS", "HINDUNILVR.NS", "NESTLEIND.NS", "BRITANNIA.NS", "TATACONSUM.NS"]
}

def run_4pillar_subfield_scanner(field_name: str = "Auto - 2-Wheelers") -> tuple:
    """Evaluates all companies in a granular sub-field across the 4 quantitative pillars."""
    tickers = SECTOR_NICHE_MAP.get(field_name, SECTOR_NICHE_MAP["Auto - 2-Wheelers"])
    print(f"[*] Scanning {len(tickers)} companies in niche field: '{field_name}'...")
    
    results = []
    historical_dfs = {}
    
    for ticker in tickers:
        try:
            t = yf.Ticker(ticker)
            hist = t.history(period="1y")
            if hist.empty or len(hist) < 50:
                continue
                
            historical_dfs[ticker] = hist
            close = float(hist['Close'].iloc[-1])
            ma_20 = float(hist['Close'].rolling(20).mean().iloc[-1])
            ma_50 = float(hist['Close'].rolling(50).mean().iloc[-1])
            ma_200 = float(hist['Close'].rolling(200).mean().iloc[-1]) if len(hist) >= 200 else ma_50
            
            # Wilder's RSI (14)
            delta = hist['Close'].diff()
            gain = delta.clip(lower=0)
            loss = -delta.clip(upper=0)
            avg_gain = gain.ewm(alpha=1/14, min_periods=14, adjust=False).mean()
            avg_loss = loss.ewm(alpha=1/14, min_periods=14, adjust=False).mean()
            rs = avg_gain / avg_loss.replace(0, np.nan)
            rsi = float((100 - (100 / (1 + rs))).iloc[-1])
            
            # Average True Range (14)
            tr = pd.concat([
                hist['High'] - hist['Low'],
                (hist['High'] - hist['Close'].shift()).abs(),
                (hist['Low'] - hist['Close'].shift()).abs()
            ], axis=1).max(axis=1)
            atr = float(tr.rolling(14).mean().iloc[-1])
            
            # Pillar 1: Financial Health (0-100)
            info = t.info or {}
            de = float(info.get('debtToEquity') or 25.0)
            roe = float((info.get('returnOnEquity') or 0.15) * 100)
            op_margin = float((info.get('operatingMargins') or 0.12) * 100)
            
            fund_score = 50.0
            if de < 50: fund_score += 20
            elif de < 100: fund_score += 10
            if roe > 15: fund_score += 15
            if op_margin > 12: fund_score += 15
            fund_score = min(100.0, max(10.0, fund_score))
            
            # Pillar 2: Technical Trend Momentum (0-100)
            tech_score = 50.0
            if close > ma_50: tech_score += 20
            if ma_50 > ma_200: tech_score += 15
            if close > ma_20: tech_score += 15
            tech_score = min(100.0, max(10.0, tech_score))
            
            # Pillar 3: RSI Momentum Sweet-Spot (0-100)
            if 45 <= rsi <= 65:
                rsi_score = 90.0
            elif rsi < 35:
                rsi_score = 80.0
            elif rsi > 70:
                rsi_score = 40.0
            else:
                rsi_score = 60.0
                
            # Pillar 4: W.D. Gann Geometric Floor Proximity (0-100)
            sqrt_p = math.sqrt(close)
            sq9_support = (sqrt_p - 0.25) ** 2
            sq9_dist = abs((close - sq9_support) / close) * 100
            gann_score = 90.0 if sq9_dist < 1.5 else (80.0 if sq9_dist < 3.0 else 65.0)
            
            # Weighted 4-Pillar Swing Index
            fund_contrib = 0.25 * fund_score
            tech_contrib = 0.30 * tech_score
            rsi_contrib = 0.20 * rsi_score
            gann_contrib = 0.25 * gann_score
            swing_index = fund_contrib + tech_contrib + rsi_contrib + gann_contrib
            
            target = round(close + (2.5 * atr), 2)
            stop_loss = round(close - (1.5 * atr), 2)
            target_pct = round(((target - close) / close) * 100, 1)
            sl_pct = round(((close - stop_loss) / close) * 100, 1)
            rrr = round((target - close) / (close - stop_loss), 1) if (close - stop_loss) > 0 else 1.7
            
            verdict = "STRONG BUY" if swing_index >= 82 else ("BUY ON DIP" if swing_index >= 70 else "NEUTRAL / WATCH")
            
            results.append({
                'Ticker': ticker,
                'Company': ticker.replace('.NS', ''),
                'CMP': round(close, 2),
                'Swing_Index': round(swing_index, 1),
                'Action': verdict,
                'Target_Price': target,
                'Target_Gain_%': f"+{target_pct}%",
                'Stop_Loss': stop_loss,
                'Risk_Loss_%': f"-{sl_pct}%",
                'RR_Ratio': f"1:{rrr}",
                'RSI_14': round(rsi, 1),
                'Fund_Pts': round(fund_contrib, 1),
                'Tech_Pts': round(tech_contrib, 1),
                'RSI_Pts': round(rsi_contrib, 1),
                'Gann_Pts': round(gann_contrib, 1)
            })
        except Exception as err:
            print(f"[!] Error scanning {ticker}: {err}")
            
    df_ranked = pd.DataFrame(results).sort_values(by='Swing_Index', ascending=False).reset_index(drop=True)
    df_ranked.index = [f"#{i+1}" for i in range(len(df_ranked))]
    df_ranked.index.name = "Rank"
    return df_ranked, historical_dfs

# 1. Run the scan
SELECTED_FIELD = "Adani Group - Top 6 Flagships"
df_opportunities, hist_dict = run_4pillar_subfield_scanner(SELECTED_FIELD)

print("========================================================================================")
print(f"🎯 ALTAIR STE: Ranked Swing Opportunity Leaderboard for '{SELECTED_FIELD}'")
print("========================================================================================")
display(df_opportunities[['Company', 'CMP', 'Swing_Index', 'Action', 'Target_Price', 'Target_Gain_%', 'Stop_Loss', 'Risk_Loss_%', 'RR_Ratio', 'RSI_14']])

# 2. Render Multi-Chart Visual Graphical Representation in Plotly
fig_field = make_subplots(
    rows=2, cols=1,
    shared_xaxes=False,
    vertical_spacing=0.15,
    row_heights=[0.55, 0.45],
    subplot_titles=(
        f"4-Pillar Quantitative Score Breakdown for {SELECTED_FIELD}",
        f"Trade Execution Risk/Reward Brackets (Target vs CMP vs Stop-Loss)"
    )
)

# Subplot 1: Stacked Bar Chart of 4 Pillars
companies = df_opportunities['Company'].tolist()
fig_field.add_trace(go.Bar(
    name='Fundamentals (25%)', y=companies, x=df_opportunities['Fund_Pts'],
    orientation='h', marker=dict(color='#10b981')
), row=1, col=1)

fig_field.add_trace(go.Bar(
    name='Trend Momentum (30%)', y=companies, x=df_opportunities['Tech_Pts'],
    orientation='h', marker=dict(color='#3b82f6')
), row=1, col=1)

fig_field.add_trace(go.Bar(
    name='RSI Sweet-Spot (20%)', y=companies, x=df_opportunities['RSI_Pts'],
    orientation='h', marker=dict(color='#a855f7')
), row=1, col=1)

fig_field.add_trace(go.Bar(
    name='Gann Floor Proximity (25%)', y=companies, x=df_opportunities['Gann_Pts'],
    orientation='h', marker=dict(color='#f59e0b')
), row=1, col=1)

# Subplot 2: Trade Execution Target & Stop-Loss Bullet Chart
for idx, row in df_opportunities.iterrows():
    c_name = row['Company']
    cmp_val = row['CMP']
    tgt_val = row['Target_Price']
    sl_val = row['Stop_Loss']
    
    # Gain range (CMP to Target in Green)
    fig_field.add_trace(go.Scatter(
        x=[cmp_val, tgt_val], y=[c_name, c_name],
        mode='lines+markers',
        line=dict(color='#22c55e', width=4),
        marker=dict(size=[8, 12], symbol=['circle', 'triangle-right'], color='#22c55e'),
        showlegend=False
    ), row=2, col=1)
    
    # Risk range (Stop-Loss to CMP in Red)
    fig_field.add_trace(go.Scatter(
        x=[sl_val, cmp_val], y=[c_name, c_name],
        mode='lines+markers',
        line=dict(color='#ef4444', width=4),
        marker=dict(size=[12, 8], symbol=['triangle-left', 'circle'], color=['#ef4444', '#22c55e']),
        showlegend=False
    ), row=2, col=1)

fig_field.update_layout(
    barmode='stack',
    title=f"🏛️ ALTAIR STE: Quantitative Analysis & Trade Risk Brackets — {SELECTED_FIELD}",
    height=800,
    template="plotly_dark",
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="center", x=0.5)
)
fig_field.show()




[*] Scanning 6 companies in niche field: 'Adani Group - Top 6 Flagships'...


🎯 ALTAIR STE: Ranked Swing Opportunity Leaderboard for 'Adani Group - Top 6 Flagships'


,Company,CMP,Swing_Index,Action,Target_Price,Target_Gain_%,Stop_Loss,Risk_Loss_%,RR_Ratio,RSI_14
Rank,,,,,,,,,,
#1,ADANIPORTS,1593.10,76.8,BUY ON DIP,1676.72,+5.2%,1542.92,-3.1%,1:1.7,30.9
#2,ADANIGREEN,1214.90,74.2,BUY ON DIP,1304.58,+7.4%,1161.09,-4.4%,1:1.7,29.2
#3,AWL,192.55,72.8,BUY ON DIP,205.40,+6.7%,184.84,-4.0%,1:1.7,49.2
#4,ATGL,621.20,70.5,BUY ON DIP,659.16,+6.1%,598.42,-3.7%,1:1.7,31.0
#5,ADANIENT,2859.10,66.5,NEUTRAL / WATCH,3059.51,+7.0%,2738.85,-4.2%,1:1.7,35.2
#6,ADANIPOWER,198.00,66.5,NEUTRAL / WATCH,211.85,+7.0%,189.69,-4.2%,1:1.7,36.7


# 🪙 SECTION 10: PRECIOUS METALS QUANT LAB (GOLD & SILVER)
A dedicated quantitative research section engineered exclusively for **Gold and Silver**.

### 🎯 The Commodity Evaluation Model:
1. **Macro Valuation (Gold-to-Silver Ratio - GSR)**: Measures relative valuation between Gold and Silver. When GSR $> 80$, Silver is historically undervalued and primed for high-beta catch-up rallies.
2. **Trend & Moving Average Ribbons (20, 50, 200 DMA)**: Identifies institutional accumulation and multi-month bull regimes.
3. **Wilder's RSI(14)**: Detects momentum exhaustion, overbought cooling, and oversold rebound zones.
4. **W.D. Gann Geometric Square of 9**: Calculates mathematical harmonic resistance ceilings and support floors for both Gold and Silver.
5. **Dynamic Volatility Brackets (ATR)**: Tailored targets and risk stops accounting for Silver's $2\times-3\times$ higher volatility beta.

In [11]:
import yfinance as yf
import pandas as pd
import numpy as np
import math
import plotly.graph_objects as go
from plotly.subplots import make_subplots

print("[*] Downloading 2-year daily history for Precious Metals & Currency...")

# Ingest Global Futures & Indian ETFs
metal_tickers = ["GC=F", "SI=F", "GOLDBEES.NS", "SILVERBEES.NS", "DX-Y.NYB"]
raw_metals = yf.download(metal_tickers, period="2y", interval="1d", progress=False)

if isinstance(raw_metals.columns, pd.MultiIndex):
    closes = raw_metals['Close']
    highs = raw_metals['High']
    lows = raw_metals['Low']
else:
    closes = raw_metals

# Clean up dataframes
df_gold = pd.DataFrame({
    'Close': closes['GC=F'].dropna(),
    'High': highs['GC=F'].dropna(),
    'Low': lows['GC=F'].dropna()
}).dropna()

df_silver = pd.DataFrame({
    'Close': closes['SI=F'].dropna(),
    'High': highs['SI=F'].dropna(),
    'Low': lows['SI=F'].dropna()
}).dropna()

# 1. Calculate Gold-to-Silver Ratio (GSR)
aligned_closes = pd.concat([df_gold['Close'], df_silver['Close']], axis=1, join='inner')
aligned_closes.columns = ['Gold', 'Silver']
aligned_closes['GSR'] = aligned_closes['Gold'] / aligned_closes['Silver']
aligned_closes['GSR_MA20'] = aligned_closes['GSR'].rolling(20).mean()
aligned_closes['GSR_MA50'] = aligned_closes['GSR'].rolling(50).mean()

# 2. Calculate Indicators for Gold
for df, name in [(df_gold, 'Gold'), (df_silver, 'Silver')]:
    df['MA_20'] = df['Close'].rolling(20).mean()
    df['MA_50'] = df['Close'].rolling(50).mean()
    df['MA_200'] = df['Close'].rolling(200).mean()
    
    # RSI(14)
    delta = df['Close'].diff()
    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)
    avg_gain = gain.ewm(alpha=1/14, min_periods=14, adjust=False).mean()
    avg_loss = loss.ewm(alpha=1/14, min_periods=14, adjust=False).mean()
    rs = avg_gain / avg_loss.replace(0, np.nan)
    df['RSI'] = 100 - (100 / (1 + rs))
    
    # ATR(14)
    tr = pd.concat([
        df['High'] - df['Low'],
        (df['High'] - df['Close'].shift()).abs(),
        (df['Low'] - df['Close'].shift()).abs()
    ], axis=1).max(axis=1)
    df['ATR'] = tr.rolling(14).mean()

# 3. Compute Gann Square of 9 Levels
def get_gann_sq9_levels(price):
    sqrt_p = math.sqrt(price)
    return {
        'R_180': round((sqrt_p + 1.0) ** 2, 2),
        'R_90': round((sqrt_p + 0.5) ** 2, 2),
        'R_45': round((sqrt_p + 0.25) ** 2, 2),
        'S_45': round((sqrt_p - 0.25) ** 2, 2),
        'S_90': round((sqrt_p - 0.5) ** 2, 2),
        'S_180': round((sqrt_p - 1.0) ** 2, 2)
    }

gold_cmp = float(df_gold['Close'].iloc[-1])
silver_cmp = float(df_silver['Close'].iloc[-1])
current_gsr = float(aligned_closes['GSR'].iloc[-1])

gold_sq9 = get_gann_sq9_levels(gold_cmp)
silver_sq9 = get_gann_sq9_levels(silver_cmp)

# 4. Precious Metals Swing Scorecard & Action Table
metals_summary = [
    {
        'Asset': 'COMEX Gold (USD/oz)',
        'Symbol': 'GC=F',
        'CMP': f"${gold_cmp:.2f}",
        'Trend_Regime': 'BULLISH' if gold_cmp > df_gold['MA_50'].iloc[-1] else 'CORRECTION',
        'RSI_14': round(float(df_gold['RSI'].iloc[-1]), 1),
        'Gann_Next_Resist (+90°)': f"${gold_sq9['R_90']}",
        'Gann_Key_Floor (-90°)': f"${gold_sq9['S_90']}",
        'Target_Price (2.5x ATR)': round(gold_cmp + (2.5 * float(df_gold['ATR'].iloc[-1])), 2),
        'Stop_Loss (1.5x ATR)': round(gold_cmp - (1.5 * float(df_gold['ATR'].iloc[-1])), 2),
        'Action': 'BUY ON DIP' if df_gold['RSI'].iloc[-1] < 55 else 'HOLD / EXTENDED'
    },
    {
        'Asset': 'COMEX Silver (USD/oz)',
        'Symbol': 'SI=F',
        'CMP': f"${silver_cmp:.2f}",
        'Trend_Regime': 'BULLISH' if silver_cmp > df_silver['MA_50'].iloc[-1] else 'CORRECTION',
        'RSI_14': round(float(df_silver['RSI'].iloc[-1]), 1),
        'Gann_Next_Resist (+90°)': f"${silver_sq9['R_90']}",
        'Gann_Key_Floor (-90°)': f"${silver_sq9['S_90']}",
        'Target_Price (2.5x ATR)': round(silver_cmp + (2.5 * float(df_silver['ATR'].iloc[-1])), 2),
        'Stop_Loss (1.5x ATR)': round(silver_cmp - (1.5 * float(df_silver['ATR'].iloc[-1])), 2),
        'Action': 'STRONG BUY (High Beta)' if current_gsr > 75 else 'BUY ON DIP'
    }
]

df_metals_scorecard = pd.DataFrame(metals_summary)

print("========================================================================================")
print(f"🪙 PRECIOUS METALS QUANT DASHBOARD | Current Gold-to-Silver Ratio: {current_gsr:.2f}")
print("========================================================================================")
display(df_metals_scorecard[['Asset', 'CMP', 'Trend_Regime', 'RSI_14', 'Gann_Key_Floor (-90°)', 'Gann_Next_Resist (+90°)', 'Target_Price (2.5x ATR)', 'Stop_Loss (1.5x ATR)', 'Action']])

# 5. Build 3-Panel Visual Precious Metals Chart in Plotly
fig_metals = make_subplots(
    rows=3, cols=1,
    shared_xaxes=True,
    vertical_spacing=0.04,
    row_heights=[0.38, 0.38, 0.24],
    subplot_titles=(
        "COMEX Gold (USD/oz) with 50/200 DMA Ribbon & Gann Square of 9 Levels",
        "COMEX Silver (USD/oz) with 50/200 DMA Ribbon & High-Beta ATR Volatility",
        f"The Historic Gold-to-Silver Ratio (GSR): {current_gsr:.1f} (High = Silver Undervalued)"
    )
)

# Panel 1: Gold Price
fig_metals.add_trace(go.Scatter(
    x=df_gold.index, y=df_gold['Close'],
    mode='lines', line=dict(color='#fbbf24', width=2.5), name="Gold"
), row=1, col=1)
fig_metals.add_trace(go.Scatter(
    x=df_gold.index, y=df_gold['MA_50'],
    mode='lines', line=dict(color='#3b82f6', width=1.5, dash='dot'), name="Gold 50 DMA"
), row=1, col=1)
fig_metals.add_trace(go.Scatter(
    x=df_gold.index, y=df_gold['MA_200'],
    mode='lines', line=dict(color='#ef4444', width=1.5), name="Gold 200 DMA"
), row=1, col=1)

# Add Gann Square of 9 levels on Gold
fig_metals.add_hline(y=gold_sq9['R_90'], line_dash="dash", line_color="rgba(239, 68, 68, 0.5)", annotation_text="Gann +90° R", row=1, col=1)
fig_metals.add_hline(y=gold_sq9['S_90'], line_dash="dash", line_color="rgba(34, 197, 94, 0.5)", annotation_text="Gann -90° S", row=1, col=1)

# Panel 2: Silver Price
fig_metals.add_trace(go.Scatter(
    x=df_silver.index, y=df_silver['Close'],
    mode='lines', line=dict(color='#e2e8f0', width=2.5), name="Silver"
), row=2, col=1)
fig_metals.add_trace(go.Scatter(
    x=df_silver.index, y=df_silver['MA_50'],
    mode='lines', line=dict(color='#3b82f6', width=1.5, dash='dot'), name="Silver 50 DMA"
), row=2, col=1)
fig_metals.add_trace(go.Scatter(
    x=df_silver.index, y=df_silver['MA_200'],
    mode='lines', line=dict(color='#ef4444', width=1.5), name="Silver 200 DMA"
), row=2, col=1)

# Add Gann Square of 9 levels on Silver
fig_metals.add_hline(y=silver_sq9['R_90'], line_dash="dash", line_color="rgba(239, 68, 68, 0.5)", annotation_text="Gann +90° R", row=2, col=1)
fig_metals.add_hline(y=silver_sq9['S_90'], line_dash="dash", line_color="rgba(34, 197, 94, 0.5)", annotation_text="Gann -90° S", row=2, col=1)

# Panel 3: Gold-to-Silver Ratio (GSR)
fig_metals.add_trace(go.Scatter(
    x=aligned_closes.index, y=aligned_closes['GSR'],
    mode='lines', line=dict(color='#a855f7', width=2), name="Gold/Silver Ratio (GSR)"
), row=3, col=1)
fig_metals.add_hline(y=80, line_dash="dot", line_color="#22c55e", annotation_text="Silver Extreme Undervaluation (>80)", row=3, col=1)
fig_metals.add_hline(y=65, line_dash="dot", line_color="#ef4444", annotation_text="Gold Defensive Sweet-Spot (<65)", row=3, col=1)

fig_metals.update_layout(
    title="🪙 ALTAIR PRECIOUS METALS LAB: Gold, Silver & Macro Ratio Harmonics",
    xaxis_rangeslider_visible=False,
    height=950,
    template="plotly_dark",
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="center", x=0.5)
)
fig_metals.show()

[*] Downloading 2-year daily history for Precious Metals & Currency...


🪙 PRECIOUS METALS QUANT DASHBOARD | Current Gold-to-Silver Ratio: 66.97


,Asset,CMP,Trend_Regime,RSI_14,Gann_Key_Floor (-90°),Gann_Next_Resist (+90°),Target_Price (2.5x ATR),Stop_Loss (1.5x ATR),Action
0,COMEX Gold (USD/oz),$4479.90,BULLISH,57.1,$4413.22,$4547.08,4674.60,4363.08,HOLD / EXTENDED
1,COMEX Silver (USD/oz),$66.89,BULLISH,56.0,$58.96,$75.32,71.81,63.94,BUY ON DIP


# 💎 SECTION 11: VALUE-QUALITY QUANT SCANNER (FMCG, BFSI & PHARMA)
Focused exclusively on **India's 3 Highest-Compounding Domestic Sectors: FMCG, BFSI & Pharma** (filtering out IT services to avoid global tech capex stagnation).

### 🏛️ Why FMCG, BFSI & Pharma for Smooth Swings:
1. **BFSI (Domestic Credit Supercycle)**: Multi-year low NPAs, credit expansion, and lowest institutional P/E multiples ($6\times - 15\times$).
2. **Pharma (Resilient Margins & Global Formulations)**: Clean balance sheets, high ROE ($>20\%$) and stable US generic pricing.
3. **FMCG (Rural Demand Rebound & Pricing Power)**: Massive free cash flow yields, dividend safety, and deeply discounted pullbacks ($20\% - 35\%$ from peaks).


In [12]:
import yfinance as yf
import pandas as pd
import numpy as np
import math
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Curated high-growth domestic leaders in FMCG, BFSI & Pharma
MACRO_UNIVERSE = {
    "Pharma": ["SUNPHARMA.NS", "CIPLA.NS", "DRREDDY.NS", "DIVISLAB.NS", "LUPIN.NS", "TORNTPHARM.NS", "MANKIND.NS", "ZYDUSLIFE.NS"],
    "BFSI": ["HDFCBANK.NS", "ICICIBANK.NS", "KOTAKBANK.NS", "AXISBANK.NS", "SBIN.NS", "BANKBARODA.NS", "BAJFINANCE.NS", "CHOLAFIN.NS"],
    "FMCG": ["ITC.NS", "HINDUNILVR.NS", "NESTLEIND.NS", "BRITANNIA.NS", "TATACONSUM.NS", "DABUR.NS", "MARICO.NS", "COLPAL.NS"]
}

print("[*] Scanning FMCG, BFSI & Pharma leaders for Deep Value & Smooth Swing Rebounds...")

macro_results = []

for sector, tickers in MACRO_UNIVERSE.items():
    for ticker in tickers:
        try:
            t = yf.Ticker(ticker)
            hist = t.history(period="1y")
            if hist.empty or len(hist) < 50:
                continue
                
            close = float(hist['Close'].iloc[-1])
            high_52w = float(hist['High'].max())
            discount_ath = ((high_52w - close) / high_52w) * 100
            
            ma_20 = float(hist['Close'].rolling(20).mean().iloc[-1])
            ma_50 = float(hist['Close'].rolling(50).mean().iloc[-1])
            ma_200 = float(hist['Close'].rolling(200).mean().iloc[-1]) if len(hist) >= 200 else ma_50
            
            # RSI(14)
            delta = hist['Close'].diff()
            gain = delta.clip(lower=0)
            loss = -delta.clip(upper=0)
            avg_gain = gain.ewm(alpha=1/14, min_periods=14, adjust=False).mean()
            avg_loss = loss.ewm(alpha=1/14, min_periods=14, adjust=False).mean()
            rs = avg_gain / avg_loss.replace(0, np.nan)
            rsi = float((100 - (100 / (1 + rs))).iloc[-1])
            
            # ATR(14)
            tr = pd.concat([
                hist['High'] - hist['Low'],
                (hist['High'] - hist['Close'].shift()).abs(),
                (hist['Low'] - hist['Close'].shift()).abs()
            ], axis=1).max(axis=1)
            atr = float(tr.rolling(14).mean().iloc[-1])
            
            # Fundamentals
            info = t.info or {}
            pe = float(info.get('trailingPE') or info.get('forwardPE') or 28.0)
            de = float(info.get('debtToEquity') or 20.0)
            roe = float((info.get('returnOnEquity') or 0.16) * 100)
            op_margin = float((info.get('operatingMargins') or 0.18) * 100)
            
            # 1. Quality Score (40%)
            q_score = 50.0
            if sector == "BFSI":
                if roe > 16: q_score += 25
                elif roe > 12: q_score += 15
                if pe < 16: q_score += 25
                elif pe < 22: q_score += 15
            else:
                if de < 25: q_score += 25
                elif de < 50: q_score += 15
                elif de > 100: q_score -= 20
                if roe > 20: q_score += 15
                elif roe > 14: q_score += 10
                if op_margin > 20: q_score += 15
                elif op_margin > 14: q_score += 10
            q_score = min(100.0, max(10.0, q_score))
            
            # 2. Valuation Score (35%)
            v_score = 40.0
            if 12 <= discount_ath <= 35:
                v_score += 35
            elif 6 <= discount_ath < 12:
                v_score += 20
            elif discount_ath > 35:
                v_score += 15
                
            if pe < 25: v_score += 25
            elif pe < 38: v_score += 15
            v_score = min(100.0, max(10.0, v_score))
            
            # 3. Smooth Swing Base Score (25%)
            m_score = 50.0
            if 35 <= rsi <= 55:
                m_score += 25
            elif rsi < 35:
                m_score += 20
            elif rsi > 65:
                m_score -= 15
            if close > ma_20:
                m_score += 15
                
            # Gann Square of 9 Floor
            sqrt_p = math.sqrt(close)
            sq9_floor = (sqrt_p - 0.25) ** 2
            sq9_dist = abs((close - sq9_floor) / close) * 100
            if sq9_dist < 2.0:
                m_score += 10
            m_score = min(100.0, max(10.0, m_score))
            
            composite_score = (0.40 * q_score) + (0.35 * v_score) + (0.25 * m_score)
            
            target = round(close + (2.5 * atr), 2)
            stop_loss = round(close - (1.5 * atr), 2)
            target_pct = round(((target - close) / close) * 100, 1)
            sl_pct = round(((close - stop_loss) / close) * 100, 1)
            rrr = round((target - close) / (close - stop_loss), 1) if (close - stop_loss) > 0 else 1.7
            
            verdict = "PRIME VALUE SWING" if composite_score >= 88 else ("QUALITY ACCUMULATE" if composite_score >= 75 else "WATCH")
            
            macro_results.append({
                'Sector': sector,
                'Company': ticker.replace('.NS', ''),
                'CMP': round(close, 2),
                'Value_Score': round(composite_score, 1),
                'Verdict': verdict,
                'Target_Price': target,
                'Target_Gain_%': f"+{target_pct}%",
                'Stop_Loss': stop_loss,
                'Risk_Buffer_%': f"-{sl_pct}%",
                'RR_Ratio': f"1:{rrr}",
                'RSI_14': round(rsi, 1),
                'Pullback_from_ATH': f"-{round(discount_ath, 1)}%",
                'PE_Ratio': round(pe, 1),
                'ROE_%': f"{round(roe, 1)}%",
                'Quality_Rating': round(q_score, 0),
                'Valuation_Discount': round(v_score, 0)
            })
        except Exception as e:
            pass

df_macro_ranked = pd.DataFrame(macro_results).sort_values(by='Value_Score', ascending=False).reset_index(drop=True)
df_macro_ranked.index = [f"#{i+1}" for i in range(len(df_macro_ranked))]
df_macro_ranked.index.name = "Rank"

print("========================================================================================")
print("💎 TOP UNDERVALUED QUALITY SWING OPPORTUNITIES (FMCG, BFSI & PHARMA)")
print("========================================================================================")
display(df_macro_ranked[['Company', 'Sector', 'CMP', 'Value_Score', 'Verdict', 'Target_Price', 'Target_Gain_%', 'Stop_Loss', 'Risk_Buffer_%', 'RR_Ratio', 'RSI_14', 'Pullback_from_ATH', 'PE_Ratio']].head(10))

# 2D Golden Quadrant Matrix (Color Coded by Sector)
top10 = df_macro_ranked.head(10)
fig_macro = make_subplots(
    rows=2, cols=1,
    vertical_spacing=0.15,
    row_heights=[0.55, 0.45],
    subplot_titles=(
        "The Golden Quadrant: Valuation Discount vs. Financial Quality (by Sector)",
        "Top 10 Swing Setups: Upside Targets vs Downside Risk Stops"
    )
)

sector_colors = {"BFSI": "#f59e0b", "Pharma": "#10b981", "FMCG": "#06b6d4"}

for sec_name, group in top10.groupby('Sector'):
    fig_macro.add_trace(go.Scatter(
        x=group['Valuation_Discount'],
        y=group['Quality_Rating'],
        mode='markers+text',
        marker=dict(
            size=[v * 0.35 for v in group['Value_Score']],
            color=sector_colors.get(sec_name, '#a855f7')
        ),
        text=group['Company'],
        textposition="top center",
        name=sec_name
    ), row=1, col=1)

# Risk/Reward Bullet Bars
for idx, r in top10.iterrows():
    c = r['Company']
    cmp_val = r['CMP']
    tgt = r['Target_Price']
    sl = r['Stop_Loss']
    
    fig_macro.add_trace(go.Scatter(
        x=[cmp_val, tgt], y=[c, c],
        mode='lines+markers',
        line=dict(color='#22c55e', width=4),
        marker=dict(size=[7, 11], symbol=['circle', 'triangle-right'], color='#22c55e'),
        showlegend=False
    ), row=2, col=1)
    
    fig_macro.add_trace(go.Scatter(
        x=[sl, cmp_val], y=[c, c],
        mode='lines+markers',
        line=dict(color='#ef4444', width=4),
        marker=dict(size=[11, 7], symbol=['triangle-left', 'circle'], color=['#ef4444', '#22c55e']),
        showlegend=False
    ), row=2, col=1)

fig_macro.update_layout(
    title="💎 ALTAIR STE: FMCG, BFSI & Pharma Value-Quality Leaderboard",
    height=850,
    template="plotly_dark",
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="center", x=0.5)
)
fig_macro.update_xaxes(title_text="Valuation Discount Score (Higher = Cheaper)", row=1, col=1)
fig_macro.update_yaxes(title_text="Financial Quality Score (Higher = Stronger Balance Sheet)", row=1, col=1)
fig_macro.show()



[*] Scanning FMCG, BFSI & Pharma leaders for Deep Value & Smooth Swing Rebounds...


💎 TOP UNDERVALUED QUALITY SWING OPPORTUNITIES (FMCG, BFSI & PHARMA)


,Company,Sector,CMP,Value_Score,Verdict,Target_Price,Target_Gain_%,Stop_Loss,Risk_Buffer_%,RR_Ratio,RSI_14,Pullback_from_ATH,PE_Ratio
Rank,,,,,,,,,,,,,
#1,SBIN,BFSI,1060.0,96.0,PRIME VALUE SWING,1098.00,+3.6%,1037.20,-2.2%,1:1.7,54.6,-12.6%,11.3
#2,LUPIN,Pharma,2188.4,91.0,PRIME VALUE SWING,2280.19,+4.2%,2133.33,-2.5%,1:1.7,34.4,-13.3%,18.1
#3,HDFCBANK,BFSI,709.0,91.0,PRIME VALUE SWING,735.86,+3.8%,692.89,-2.3%,1:1.7,33.2,-29.4%,15.5
#4,COLPAL,FMCG,1897.6,87.5,QUALITY ACCUMULATE,1984.53,+4.6%,1845.44,-2.7%,1:1.7,41.2,-22.5%,38.3
#5,DABUR,FMCG,381.0,87.0,QUALITY ACCUMULATE,394.64,+3.6%,372.81,-2.1%,1:1.7,22.0,-32.8%,34.3
#6,CIPLA,Pharma,1409.2,86.8,QUALITY ACCUMULATE,1459.72,+3.6%,1378.89,-2.2%,1:1.7,41.3,-15.0%,33.8
#7,HINDUNILVR,FMCG,1967.4,86.2,QUALITY ACCUMULATE,2034.40,+3.4%,1927.20,-2.0%,1:1.7,31.2,-27.2%,43.7
#8,ITC,FMCG,255.5,85.5,QUALITY ACCUMULATE,266.71,+4.4%,248.77,-2.6%,1:1.7,24.1,-37.2%,16.1
#9,AXISBANK,BFSI,1300.0,84.5,QUALITY ACCUMULATE,1350.64,+3.9%,1269.61,-2.3%,1:1.7,64.4,-8.3%,14.6
